# 16장 이미지 분류와 멀티모달 실습

이 장은 PDF 교재의 Hugging Face Pipeline, Gradio, LLM 응용 실습 흐름을 바탕으로 이미지 데이터를 다루는 방법을 정리합니다.

앞 장들에서는 텍스트 질문, 문서 검색, DB 조회, 음성 입출력을 다뤘습니다. 이번 장에서는 이미지 분류 모델을 사용해 이미지를 입력받고, 모델이 예측한 결과를 텍스트로 설명하거나 Gradio UI에서 확인하는 방법을 다룹니다.

## 이 장에서 다루는 내용

1. 이미지 분류와 멀티모달 개념
2. 텍스트 모델과 이미지 모델의 차이
3. Hugging Face 이미지 분류 Pipeline
4. PIL을 사용한 이미지 파일 열기
5. 이미지 분류 결과 해석
6. Gradio 이미지 분류 UI
7. 이미지 설명을 LLM 답변에 연결하는 방법
8. 멀티모달 챗봇 구조
9. 이미지 분류 실습 시 주의할 점
10. 오류 해결과 확장 방향

이 장은 `LLM/llm.ipynb`의 Hugging Face Pipeline 실습과 Gradio 챗봇 실습을 이미지 입력으로 확장하는 내용입니다.


## 16.1 이미지 분류와 멀티모달 개념

이미지 분류는 입력 이미지가 어떤 범주에 속하는지 예측하는 작업입니다.

예를 들어 고양이 사진을 입력하면 모델이 다음과 같은 결과를 반환할 수 있습니다.

```text
tabby cat: 0.82
tiger cat: 0.10
Egyptian cat: 0.05
```

멀티모달은 여러 종류의 데이터를 함께 다루는 방식을 의미합니다.

| 입력 형태 | 예시 |
|---|---|
| 텍스트 | 질문, 문서, 채팅 메시지 |
| 이미지 | 사진, 도표, 스크린샷 |
| 음성 | 녹음 파일, 마이크 입력 |
| 표 데이터 | CSV, DB 조회 결과 |

LLM 실습에서 멀티모달을 다룬다는 것은 텍스트뿐 아니라 이미지나 음성 같은 입력을 함께 연결하는 것을 의미합니다.


## 16.2 텍스트 모델과 이미지 모델의 차이

텍스트 모델은 문장을 토큰으로 나누어 처리합니다. 반면 이미지 모델은 이미지를 픽셀 또는 패치 단위로 변환해 처리합니다.

| 구분 | 텍스트 모델 | 이미지 모델 |
|---|---|---|
| 입력 | 문장, 문서, 질문 | 이미지 파일, 픽셀 배열 |
| 전처리 | 토큰화 | 크기 조정, 정규화 |
| 대표 작업 | 분류, 요약, 번역, 질의응답 | 이미지 분류, 객체 탐지, 이미지 캡셔닝 |
| 출력 | 텍스트 또는 클래스 | 클래스, 박스 좌표, 설명 문장 |

Hugging Face의 `pipeline`은 텍스트와 이미지 작업을 비슷한 방식으로 사용할 수 있게 도와줍니다. 텍스트 분류에서는 `pipeline("text-classification")`을 쓰고, 이미지 분류에서는 `pipeline("image-classification")`을 사용할 수 있습니다.


In [ ]:
# 교재 위치: 16장 이미지 분류와 멀티모달 실습 - 설치 준비
# 이 셀은 이미지 분류와 Gradio UI 실습에 필요한 패키지를 설치합니다.
# 이미 설치되어 있다면 실행하지 않아도 됩니다.

# transformers는 Hugging Face 모델과 pipeline 기능을 사용하기 위한 라이브러리입니다.
!pip install transformers

# torch는 많은 Hugging Face 딥러닝 모델의 실행 백엔드로 사용됩니다.
!pip install torch

# pillow는 Python에서 이미지 파일을 열고 처리할 때 사용하는 라이브러리입니다.
!pip install pillow

# gradio는 이미지 입력과 결과 출력을 위한 웹 UI를 만들 때 사용합니다.
!pip install gradio

# langchain과 langchain-ollama는 이미지 분류 결과를 LLM 설명으로 연결할 때 사용합니다.
!pip install langchain langchain-ollama


## 16.3 이미지 파일 열기

이미지 분류 모델에 이미지를 넣기 전에 Python에서 이미지 파일을 열어 확인할 수 있습니다. 가장 기본적인 도구는 `PIL`, 즉 Pillow 라이브러리입니다.

실습에서는 현재 노트북 폴더에 `sample_image.jpg` 파일이 있다고 가정합니다. 파일이 없다면 임의의 이미지 파일을 준비하고 경로만 바꾸면 됩니다.


In [ ]:
# 교재 위치: 16장 이미지 분류와 멀티모달 실습 - 이미지 파일 열기
# 이 셀은 Pillow를 사용해 이미지 파일을 여는 기본 예시입니다.

# Path는 파일 경로를 객체 형태로 안전하게 다룰 때 사용합니다.
from pathlib import Path

# Image는 Pillow에서 이미지 파일을 열고 처리하는 클래스입니다.
from PIL import Image

# image_path 변수에는 실습에 사용할 이미지 파일 경로를 지정합니다.
image_path = Path("sample_image.jpg")

# exists 메서드는 지정한 이미지 파일이 실제로 존재하는지 확인합니다.
image_exists = image_path.exists()

# 이미지 파일 존재 여부를 출력합니다.
print("이미지 파일 존재 여부:", image_exists)

# 이미지 파일이 존재할 때만 파일을 엽니다.
if image_exists:
    # Image.open은 이미지 파일을 열어 PIL Image 객체로 반환합니다.
    image = Image.open(image_path)

    # image.size는 이미지의 가로와 세로 크기를 반환합니다.
    print("이미지 크기:", image.size)

    # display는 Jupyter Notebook에서 이미지를 화면에 표시합니다.
    display(image)
else:
    # 파일이 없을 때 준비해야 할 파일명을 안내합니다.
    print("현재 폴더에 sample_image.jpg 파일을 준비하면 이미지 실습을 실행할 수 있습니다.")


## 16.4 Hugging Face 이미지 분류 Pipeline

Hugging Face `pipeline`을 사용하면 이미지 분류 모델을 간단하게 불러올 수 있습니다.

기본 흐름은 다음과 같습니다.

```text
pipeline("image-classification") 생성
  -> 이미지 파일 또는 PIL Image 입력
  -> 예측 결과 리스트 반환
```

결과는 보통 다음처럼 여러 후보 클래스를 점수와 함께 반환합니다.

```python
[
    {"label": "tabby cat", "score": 0.82},
    {"label": "tiger cat", "score": 0.10},
]
```


In [ ]:
# 교재 위치: 16장 이미지 분류와 멀티모달 실습 - 이미지 분류 Pipeline
# 이 셀은 Hugging Face pipeline으로 이미지 분류 모델을 준비합니다.

# pipeline은 Hugging Face에서 작업 유형별 모델 실행기를 쉽게 만드는 함수입니다.
from transformers import pipeline

# image_classifier는 이미지 분류 작업을 수행하는 pipeline 객체입니다.
image_classifier = pipeline("image-classification")

# classify_image 함수는 이미지 파일 경로를 받아 분류 결과를 반환합니다.
def classify_image(file_path: str) -> list[dict]:
    # 파일 경로가 없으면 빈 리스트를 반환합니다.
    if file_path is None:
        # 입력 이미지가 없다는 뜻으로 빈 결과를 반환합니다.
        return []

    # image_classifier에 이미지 파일 경로를 전달해 분류를 실행합니다.
    results = image_classifier(file_path)

    # 모델이 반환한 분류 결과 리스트를 반환합니다.
    return results


# 이미지 파일이 있을 때만 분류 예제를 실행합니다.
if image_path.exists():
    # sample_image.jpg 파일을 분류합니다.
    classification_results = classify_image(str(image_path))

    # 분류 결과를 출력합니다.
    print(classification_results)


## 16.5 이미지 분류 결과 해석

이미지 분류 결과는 보통 `label`과 `score`로 구성됩니다.

| 필드 | 의미 |
|---|---|
| label | 모델이 예측한 클래스 이름 |
| score | 해당 클래스일 확률 또는 신뢰도 |

가장 높은 점수를 가진 결과를 top-1 예측이라고 부릅니다. 여러 후보를 함께 보면 모델이 어떤 클래스 사이에서 헷갈렸는지도 확인할 수 있습니다.

이미지 분류 결과를 사용자에게 그대로 보여줄 수도 있지만, LLM을 연결해 자연어 설명으로 바꿀 수도 있습니다.


In [ ]:
# 교재 위치: 16장 이미지 분류와 멀티모달 실습 - 분류 결과 정리
# 이 셀은 이미지 분류 결과를 사람이 읽기 좋은 문자열로 정리합니다.

# format_classification_results 함수는 모델 결과 리스트를 텍스트로 변환합니다.
def format_classification_results(results: list[dict], top_k: int = 5) -> str:
    # 결과가 비어 있으면 안내 문장을 반환합니다.
    if not results:
        # 분류 결과가 없을 때 사용할 문장입니다.
        return "분류 결과가 없습니다."

    # 상위 top_k개 결과만 선택합니다.
    top_results = results[:top_k]

    # formatted_lines에는 각 분류 결과를 한 줄씩 저장합니다.
    formatted_lines = []

    # enumerate는 결과 목록을 순번과 함께 반복합니다.
    for index, item in enumerate(top_results, start=1):
        # label은 예측된 클래스 이름입니다.
        label = item.get("label", "unknown")

        # score는 예측 신뢰도입니다.
        score = item.get("score", 0.0)

        # score를 퍼센트 형태로 바꿔 읽기 좋게 만듭니다.
        percent = score * 100

        # 한 줄 설명을 만들어 리스트에 추가합니다.
        formatted_lines.append(f"{index}. {label}: {percent:.2f}%")

    # 여러 줄을 줄바꿈으로 연결해 하나의 문자열로 반환합니다.
    return "\n".join(formatted_lines)


# 예시 결과가 있을 때만 정리 함수를 실행합니다.
if image_path.exists():
    # 분류 결과를 사람이 읽기 좋은 텍스트로 변환합니다.
    formatted_result_text = format_classification_results(classification_results)

    # 변환된 결과 텍스트를 출력합니다.
    print(formatted_result_text)


## 16.6 Gradio 이미지 분류 UI

Gradio의 `Image` 컴포넌트를 사용하면 사용자가 이미지를 업로드하고, 모델 분류 결과를 화면에서 바로 확인할 수 있습니다.

기본 구조는 다음과 같습니다.

```text
Image 입력
  -> classify_image 함수
  -> 결과 Textbox 출력
```

파일 경로 방식으로 처리하려면 Gradio `Image` 컴포넌트의 `type`을 `filepath`로 지정합니다.


In [ ]:
# 교재 위치: 16장 이미지 분류와 멀티모달 실습 - Gradio 이미지 분류 UI
# 이 셀은 이미지를 업로드하면 분류 결과를 보여주는 Gradio UI를 만듭니다.

# gradio를 gr이라는 별칭으로 불러옵니다.
import gradio as gr


# classify_for_ui 함수는 Gradio에서 받은 이미지 파일 경로를 처리합니다.
def classify_for_ui(image_file: str) -> str:
    # 이미지 파일이 없으면 안내 문장을 반환합니다.
    if image_file is None:
        # 사용자가 이미지를 업로드하지 않았을 때의 메시지입니다.
        return "이미지를 업로드해 주세요."

    # 이미지 분류 pipeline을 실행합니다.
    results = classify_image(image_file)

    # 분류 결과를 사람이 읽기 좋은 문자열로 바꿉니다.
    result_text = format_classification_results(results)

    # 변환된 결과 문자열을 반환합니다.
    return result_text


# Blocks는 여러 Gradio 컴포넌트를 배치할 수 있는 컨테이너입니다.
with gr.Blocks() as image_demo:
    # Markdown은 화면 제목을 표시합니다.
    gr.Markdown("# 이미지 분류 실습")

    # Image 컴포넌트는 이미지 업로드 입력을 받습니다.
    image_input = gr.Image(type="filepath", label="이미지 업로드")

    # Textbox는 이미지 분류 결과를 텍스트로 표시합니다.
    result_output = gr.Textbox(label="분류 결과", lines=8)

    # Button은 분류 실행 버튼입니다.
    classify_button = gr.Button("이미지 분류하기")

    # 버튼 클릭 시 classify_for_ui 함수를 실행합니다.
    classify_button.click(
        # fn에는 실행할 함수를 지정합니다.
        fn=classify_for_ui,
        # inputs에는 이미지 입력 컴포넌트를 지정합니다.
        inputs=image_input,
        # outputs에는 결과 출력 Textbox를 지정합니다.
        outputs=result_output,
    )

# launch를 실행하면 로컬 웹 브라우저에서 이미지 분류 UI를 확인할 수 있습니다.
# image_demo.launch()


## 16.7 이미지 분류 결과를 LLM 설명으로 연결하기

이미지 분류 모델은 클래스 이름과 점수를 반환합니다. 하지만 사용자는 결과가 무엇을 의미하는지 자연어 설명을 원할 수 있습니다.

이때 LLM을 연결하면 다음처럼 만들 수 있습니다.

```text
이미지 입력
  -> 이미지 분류 모델
  -> label, score 추출
  -> LLM 프롬프트에 결과 전달
  -> 자연어 설명 생성
```

주의할 점은 이미지 분류 모델이 실제 이미지를 완벽히 이해하는 것이 아니라, 학습된 클래스 중에서 가장 가까운 후보를 고른다는 것입니다. 따라서 LLM 설명에서도 모델 예측이라는 점을 분명히 하는 것이 좋습니다.


In [ ]:
# 교재 위치: 16장 이미지 분류와 멀티모달 실습 - LLM 설명 연결
# 이 셀은 이미지 분류 결과를 LLM으로 설명하는 체인을 구성합니다.

# ChatPromptTemplate은 LLM에 전달할 프롬프트 템플릿을 만듭니다.
from langchain_core.prompts import ChatPromptTemplate

# StrOutputParser는 LLM 응답을 문자열로 변환합니다.
from langchain_core.output_parsers import StrOutputParser

# ChatOllama는 Ollama 로컬 LLM을 LangChain 방식으로 호출합니다.
from langchain_ollama import ChatOllama

# llm 변수에는 사용할 Ollama 모델 이름을 지정합니다.
llm = ChatOllama(model="llama3.1")

# explanation_prompt는 이미지 분류 결과를 설명하기 위한 프롬프트입니다.
explanation_prompt = ChatPromptTemplate.from_messages([
    # system 메시지는 모델의 역할과 답변 원칙을 지정합니다.
    ("system", "너는 이미지 분류 결과를 초보자에게 쉽게 설명하는 한국어 튜터야."),
    # human 메시지는 분류 결과 텍스트를 입력으로 받습니다.
    ("human", "다음 이미지 분류 결과를 해석해줘. 모델 예측이라는 점도 함께 설명해줘.\n\n{classification}"),
])

# explanation_chain은 프롬프트, LLM, 출력 파서를 연결한 체인입니다.
explanation_chain = explanation_prompt | llm | StrOutputParser()


# explain_classification 함수는 분류 결과 텍스트를 LLM 설명으로 바꿉니다.
def explain_classification(classification_text: str) -> str:
    # 분류 결과가 비어 있으면 안내 문장을 반환합니다.
    if not classification_text:
        # 설명할 결과가 없다는 메시지입니다.
        return "설명할 이미지 분류 결과가 없습니다."

    # LLM 체인을 실행해 분류 결과에 대한 설명을 생성합니다.
    explanation = explanation_chain.invoke({"classification": classification_text})

    # 생성된 설명을 반환합니다.
    return explanation


# 예시 분류 결과 텍스트가 있을 때만 설명 예제를 실행합니다.
if image_path.exists():
    # 이미지 분류 결과를 LLM 설명으로 변환합니다.
    explanation_text = explain_classification(formatted_result_text)

    # 생성된 설명을 출력합니다.
    print(explanation_text)


## 16.8 이미지 분류와 LLM 설명을 함께 제공하는 UI

이미지 분류 결과와 LLM 설명을 함께 제공하면 사용자가 모델 출력의 의미를 더 쉽게 이해할 수 있습니다.

UI 흐름은 다음과 같습니다.

```text
Image 입력
  -> 이미지 분류
  -> 분류 결과 출력
  -> LLM 설명 생성
  -> 설명 출력
```


In [ ]:
# 교재 위치: 16장 이미지 분류와 멀티모달 실습 - 멀티모달 설명 UI
# 이 셀은 이미지 분류 결과와 LLM 설명을 함께 보여주는 Gradio UI를 만듭니다.

# classify_and_explain 함수는 이미지를 분류하고 결과를 LLM으로 설명합니다.
def classify_and_explain(image_file: str) -> tuple[str, str]:
    # 이미지 입력이 없으면 안내 문장 두 개를 반환합니다.
    if image_file is None:
        # 분류 결과와 설명 결과를 각각 안내 문장으로 반환합니다.
        return "이미지를 업로드해 주세요.", "이미지가 입력되면 설명을 생성합니다."

    # 이미지 분류 모델을 실행합니다.
    results = classify_image(image_file)

    # 분류 결과를 텍스트로 정리합니다.
    classification_text = format_classification_results(results)

    # 정리된 분류 결과를 LLM에 전달해 설명을 생성합니다.
    explanation = explain_classification(classification_text)

    # 분류 결과와 LLM 설명을 함께 반환합니다.
    return classification_text, explanation


# Blocks로 멀티모달 설명 UI를 구성합니다.
with gr.Blocks() as multimodal_demo:
    # 화면 제목을 Markdown으로 표시합니다.
    gr.Markdown("# 이미지 분류 + LLM 설명")

    # 이미지 입력 컴포넌트를 만듭니다.
    multimodal_image_input = gr.Image(type="filepath", label="이미지 업로드")

    # 분류 결과 출력 Textbox를 만듭니다.
    classification_output = gr.Textbox(label="이미지 분류 결과", lines=8)

    # LLM 설명 출력 Textbox를 만듭니다.
    explanation_output = gr.Textbox(label="LLM 설명", lines=10)

    # 실행 버튼을 만듭니다.
    explain_button = gr.Button("분류하고 설명하기")

    # 버튼 클릭 시 classify_and_explain 함수를 실행합니다.
    explain_button.click(
        # fn에는 실행할 함수를 지정합니다.
        fn=classify_and_explain,
        # inputs에는 이미지 입력 컴포넌트를 지정합니다.
        inputs=multimodal_image_input,
        # outputs에는 분류 결과와 설명 출력 컴포넌트를 지정합니다.
        outputs=[classification_output, explanation_output],
    )

# launch를 실행하면 로컬 웹 브라우저에서 멀티모달 설명 UI를 확인할 수 있습니다.
# multimodal_demo.launch()


## 16.9 멀티모달 챗봇 구조

이미지 분류 실습은 멀티모달 챗봇의 가장 단순한 형태입니다. 더 발전시키면 사용자가 이미지와 질문을 함께 입력하고, 모델이 이미지 정보와 질문을 함께 고려해 답변하는 구조를 만들 수 있습니다.

기본 구조는 다음과 같습니다.

```text
이미지 입력 + 텍스트 질문
  -> 이미지 분석 모델
  -> 이미지 분석 결과 텍스트화
  -> 질문과 이미지 분석 결과를 LLM 프롬프트에 전달
  -> 답변 생성
```

이 방식은 진정한 비전-언어 모델을 사용하는 것과는 다릅니다. 여기서는 이미지 모델이 먼저 이미지를 텍스트 정보로 바꾸고, 그 텍스트를 LLM이 사용하는 구조입니다.


In [ ]:
# 교재 위치: 16장 이미지 분류와 멀티모달 실습 - 이미지와 질문을 함께 처리
# 이 셀은 이미지 분류 결과와 사용자 질문을 함께 LLM에 전달하는 예시입니다.

# image_question_prompt는 이미지 분석 결과와 사용자 질문을 함께 받는 프롬프트입니다.
image_question_prompt = ChatPromptTemplate.from_messages([
    # system 메시지는 모델의 역할을 지정합니다.
    ("system", "너는 이미지 분류 결과와 사용자 질문을 바탕으로 답변하는 한국어 도우미야."),
    # human 메시지는 이미지 분류 결과와 질문을 템플릿 변수로 받습니다.
    ("human", "이미지 분류 결과:\n{classification}\n\n사용자 질문:\n{question}\n\n답변:"),
])

# image_question_chain은 프롬프트, LLM, 출력 파서를 연결한 체인입니다.
image_question_chain = image_question_prompt | llm | StrOutputParser()


# answer_about_image 함수는 이미지와 질문을 함께 처리합니다.
def answer_about_image(image_file: str, question: str) -> str:
    # 이미지 파일이 없으면 안내 문장을 반환합니다.
    if image_file is None:
        # 이미지가 필요하다는 안내입니다.
        return "이미지를 먼저 업로드해 주세요."

    # 질문이 비어 있으면 기본 질문을 사용합니다.
    if not question:
        # 기본 질문은 이미지가 무엇으로 분류되었는지 묻습니다.
        question = "이 이미지는 무엇으로 분류되었나요?"

    # 이미지 분류 모델을 실행합니다.
    results = classify_image(image_file)

    # 이미지 분류 결과를 문자열로 정리합니다.
    classification_text = format_classification_results(results)

    # LLM 체인에 분류 결과와 사용자 질문을 전달합니다.
    answer = image_question_chain.invoke({"classification": classification_text, "question": question})

    # 생성된 답변을 반환합니다.
    return answer


## 16.10 이미지 질문 답변 UI

이미지와 텍스트 질문을 함께 받는 UI를 만들면 멀티모달 챗봇과 비슷한 사용 경험을 만들 수 있습니다.

예시 질문은 다음과 같습니다.

- 이 이미지는 무엇으로 분류되었나요?
- 모델이 가장 확신한 클래스는 무엇인가요?
- 상위 예측 후보들이 서로 어떻게 다른가요?
- 이 결과를 사용할 때 주의할 점은 무엇인가요?


In [ ]:
# 교재 위치: 16장 이미지 분류와 멀티모달 실습 - 이미지 질문 답변 UI
# 이 셀은 이미지와 질문을 함께 입력받아 답변하는 Gradio UI를 만듭니다.

# Blocks로 이미지 질문 답변 UI를 구성합니다.
with gr.Blocks() as image_qa_demo:
    # 화면 제목을 Markdown으로 표시합니다.
    gr.Markdown("# 이미지 기반 질문 답변")

    # 이미지 입력 컴포넌트를 만듭니다.
    qa_image_input = gr.Image(type="filepath", label="이미지 업로드")

    # 질문 입력 Textbox를 만듭니다.
    qa_question_input = gr.Textbox(label="이미지에 대해 질문하기")

    # 답변 출력 Textbox를 만듭니다.
    qa_answer_output = gr.Textbox(label="답변", lines=10)

    # 실행 버튼을 만듭니다.
    qa_button = gr.Button("질문 보내기")

    # 버튼 클릭 시 answer_about_image 함수를 실행합니다.
    qa_button.click(
        # fn에는 이미지와 질문을 처리할 함수를 지정합니다.
        fn=answer_about_image,
        # inputs에는 이미지 입력과 질문 입력 컴포넌트를 지정합니다.
        inputs=[qa_image_input, qa_question_input],
        # outputs에는 답변 출력 컴포넌트를 지정합니다.
        outputs=qa_answer_output,
    )

# launch를 실행하면 로컬 웹 브라우저에서 이미지 질문 답변 UI를 확인할 수 있습니다.
# image_qa_demo.launch()


## 16.11 이미지 분류 실습 시 주의할 점

이미지 분류 결과를 해석할 때는 다음을 주의해야 합니다.

- 모델은 학습된 클래스 중에서 가장 가까운 답을 고릅니다.
- 학습 데이터에 없는 대상은 엉뚱한 클래스로 분류될 수 있습니다.
- 점수가 높다고 항상 정답이라는 뜻은 아닙니다.
- 이미지 품질, 해상도, 조명, 배경이 결과에 영향을 줍니다.
- 사람 얼굴, 의료 이미지, 법적 판단처럼 민감한 분야에는 별도 검증이 필요합니다.
- LLM이 분류 결과를 설명할 때 과장하거나 확정적으로 말하지 않도록 프롬프트를 작성해야 합니다.

교재 실습에서는 이미지 분류 결과를 모델 예측으로 이해하고, 실제 판단이 필요한 업무에서는 추가 검증 절차를 둬야 합니다.


## 16.12 자주 발생하는 오류와 해결 방법

| 오류 상황 | 원인 | 해결 방법 |
|---|---|---|
| transformers import 오류 | 패키지 미설치 | `pip install transformers` 실행 |
| torch 오류 | PyTorch 설치 문제 | Python 버전에 맞는 torch 재설치 |
| 이미지 파일 열기 실패 | 경로 오류 또는 손상된 파일 | 파일 경로와 확장자 확인 |
| 모델 다운로드가 느림 | 최초 실행 시 모델 다운로드 | 네트워크 확인 후 기다리기 |
| 분류 결과가 이상함 | 모델 클래스와 이미지가 맞지 않음 | 다른 이미지 모델 사용 또는 결과를 참고용으로 해석 |
| Gradio 이미지 입력 오류 | type 설정 불일치 | `type="filepath"` 사용 여부 확인 |
| Ollama 호출 실패 | Ollama 서버 미실행 | Ollama 실행 후 모델 이름 확인 |


## 16.13 정리

이 장에서는 이미지 분류와 멀티모달 실습의 기본 구조를 정리했습니다.

핵심 정리는 다음과 같습니다.

- 이미지 분류는 입력 이미지가 어떤 클래스에 속하는지 예측하는 작업입니다.
- Hugging Face `pipeline("image-classification")`으로 이미지 분류 모델을 쉽게 사용할 수 있습니다.
- Pillow의 `Image.open`으로 이미지 파일을 열고 확인할 수 있습니다.
- Gradio `Image` 컴포넌트로 이미지 업로드 UI를 만들 수 있습니다.
- 이미지 분류 결과는 `label`과 `score`로 구성됩니다.
- LLM을 연결하면 이미지 분류 결과를 자연어로 설명할 수 있습니다.
- 이미지와 질문을 함께 입력받으면 멀티모달 챗봇과 비슷한 구조를 만들 수 있습니다.
- 이미지 모델의 결과는 참고용 예측이며, 민감한 판단에는 추가 검증이 필요합니다.

다음 장에서는 워드클라우드와 텍스트 시각화를 다룹니다. LLM과 NLP 실습에서 생성되거나 수집된 텍스트를 시각적으로 분석하는 방법을 정리합니다.
